# U.S. State Economic Purchasing Power & Tax Data Pipeline

This notebook is the **analysis layer** of the repository. Data acquisition, tax calculations, transformations, and chart helpers live in `src/` so the notebook can focus on analytical questions, results, and interpretation.

The core output is **hypothetical after-tax purchasing power at state median household income** under explicit filing-status assumptions. It is not an estimate of actual median household tax liability.

## 1. Setup

The notebook expects to be run from the repository root. A FRED API key is required for live economic data. No credentials are stored in the repository.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import us

from src.data_collection import (
    fred_frame,
    get_fred_client,
    historical_standard_deductions,
    scrape_federal_brackets,
    ss_wage_limits,
)
from src.tax_engine import TaxEngine
from src.transformations import (
    build_purchasing_power_panel,
    linear_projection,
    prep_annual,
)
from src.visualization import (
    plot_cpi_projection,
    plot_income_vs_purchasing_power,
    plot_private_employment,
    plot_purchasing_power_trend,
    plot_real_gdp_growth,
    plot_top_purchasing_power,
    plot_top_tax_burden,
)

ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

fred = get_fred_client()

## 2. U.S. macroeconomic context

The first section establishes the macro backdrop using real GDP, potential GDP, private payroll employment, and consumer prices. Linear projections are descriptive counterfactual trend benchmarks—not causal estimates of what “should” have happened.

In [ ]:
us_rgdp = fred_frame(fred, "GDPC1", "RealGDP")
us_rgdp["Year"] = us_rgdp["Date"].dt.year

q4_real_gdp = us_rgdp.loc[us_rgdp["Date"].dt.quarter == 4].set_index("Year")["RealGDP"]
annual_change = q4_real_gdp.pct_change() * 100
annual_change.tail()

In [ ]:
us_rgdpc = fred_frame(fred, "A939RX0Q048SBEA", "RealGDPc")
us_rgdpc["Year"] = us_rgdpc["Date"].dt.year
us_rgdpc["RealGDPc"] = us_rgdpc["RealGDPc"].round(2)
us_rgdpc_year_change = (
    us_rgdpc.loc[us_rgdpc["Date"].dt.quarter == 4]
    .set_index("Year")["RealGDPc"]
    .pct_change() * 100
)
us_rgdpc_year_change.tail()

In [ ]:
potential_gdp = fred_frame(fred, "GDPPOT", "PotentialGDP")
us_cyc_rgdp = us_rgdp.merge(potential_gdp, on="Date", how="inner")
us_cyc_rgdp["CyclicalGDP"] = us_cyc_rgdp["RealGDP"] - us_cyc_rgdp["PotentialGDP"]
us_cyc_rgdp.tail()

### Cyclical GDP component proxy

The project also constructs a separate cyclical-component proxy from real durable-goods consumption, residential/equipment investment, capital-goods exports, and inventory change.

\[
Y_c = C_d + I_{res} + I_{eq} + \Delta Inv + X_{cap}
\]

In [ ]:
cyclical_components = [
    fred_frame(fred, "PCEDGC96", "rPCE_Durable_Goods", observation_end="2025-12-01"),
    fred_frame(fred, "PRFIC1", "rFixedResidentialInvestment"),
    fred_frame(fred, "Y033RX1Q020SBEA", "rEquipmentInvestment"),
    fred_frame(fred, "A640RX1A020NBEA", "rCapitalGoodsExports"),
    fred_frame(fred, "CBIC1", "rChangePrivateInventories"),
]
annual_components = pd.concat([prep_annual(frame) for frame in cyclical_components], axis=1).sort_index()
annual_components["rCyclicalGDP"] = annual_components.sum(axis=1)
cyclical_rgdp_change = annual_components.loc["2007":, "rCyclicalGDP"].pct_change() * 100
cyclical_rgdp_change.tail()

### Private payroll employment versus a pre-COVID linear trend

In [ ]:
private_emp = fred_frame(fred, "USPRIV", "Private Employees")
employment_projection = linear_projection(
    private_emp,
    y_var="Private Employees",
    period_start="2017-03-01",
    period_end="2020-02-01",
    data_type="int",
)
US_PRIV = employment_projection.copy()
US_PRIV["Difference from Projection"] = (
    US_PRIV["Private Employees"] - US_PRIV["Projected Private Employees"]
)

comparison = US_PRIV.dropna(subset=["Private Employees", "Projected Private Employees"])
latest_employment = comparison.sort_values("Date").iloc[-1]
trend_gap_jobs = float(latest_employment["Difference from Projection"]) * 1_000
trend_direction = "below" if trend_gap_jobs < 0 else "above"
print(
    f"Private payroll employment was {abs(trend_gap_jobs)/1_000_000:.2f}M jobs {trend_direction} "
    f"the extrapolated 36-month pre-COVID linear trend as of {latest_employment['Date']:%Y-%m}."
)

### Consumer prices and other trend benchmarks

In [ ]:
cpi = fred_frame(fred, "CPIAUCSL", "CPI")
cpi_projection = linear_projection(
    cpi,
    y_var="CPI",
    period_start="2017-01-01",
    period_end="2020-02-01",
    data_type="float",
)

rpce_projection = linear_projection(
    fred_frame(fred, "PCEC96", "RPCE"),
    y_var="RPCE",
    period_start="2017-01-01",
    period_end="2020-02-01",
    data_type="float",
)

pcepi_projection = linear_projection(
    fred_frame(fred, "PCEPI", "PCE"),
    y_var="PCE",
    period_start="2017-01-01",
    period_end="2020-02-01",
    data_type="float",
)

real_income_projection = linear_projection(
    fred_frame(fred, "W875RX1", "RPI"),
    y_var="RPI",
    period_start="2017-01-01",
    period_end="2020-02-01",
    data_type="float",
)

## 3. Tax engine inputs

The state-tax table is already normalized in `data/processed/`. Federal brackets/deductions and Social Security wage-base history are collected by the acquisition module, then converted into validated lookup structures by `TaxEngine`.

In [ ]:
state_tax_df = pd.read_csv(DATA_PROCESSED / "state_income_tax_master.csv")
federal_brackets_df = scrape_federal_brackets(format="df")
federal_deductions = historical_standard_deductions()
payroll_df = ss_wage_limits()

tax_engine = TaxEngine(
    federal_brackets_df=federal_brackets_df,
    federal_deductions=federal_deductions,
    payroll_df=payroll_df,
    state_tax_df=state_tax_df,
)

print(
    f"State tax coverage: {tax_engine.state_tax_min_year}-{tax_engine.state_tax_max_year} | "
    f"overlapping supported years: {min(tax_engine.supported_years)}-{max(tax_engine.supported_years)}"
)

## 4. State purchasing-power panel

For each state/year, the pipeline combines median household income (MHI), Regional Price Parity (RPP), estimated federal/payroll/state taxes, and the PCE price index.

Two adjustments are deliberately separated:

1. **RPP adjustment** compares price levels across states within a year.
2. **PCEPI adjustment** converts the resulting measure to constant 2024 dollars for comparisons over time.

In [ ]:
states = [state.abbr for state in us.states.STATES]
rpp_codes = [f"{abbr}MPRPPALL" for abbr in states]
mhi_codes = [f"MEHOINUS{abbr}A646N" for abbr in states]

married_dataset = build_purchasing_power_panel(
    fred,
    rpp_codes,
    states,
    mhi_codes,
    tax_engine,
    target_filing_status="married_joint",
    dependent_count=0,
    constant_dollar_year=2024,
)

single_dataset = build_purchasing_power_panel(
    fred,
    rpp_codes,
    states,
    mhi_codes,
    tax_engine,
    target_filing_status="single",
    dependent_count=0,
    constant_dollar_year=2024,
)

married_dataset.to_csv(DATA_PROCESSED / "state_purchasing_power_married_joint.csv", index=False)
single_dataset.to_csv(DATA_PROCESSED / "state_purchasing_power_single.csv", index=False)

married_dataset.head()

### Validation summary

In [ ]:
validation_summary = pd.DataFrame({
    "Scenario": ["single", "married_joint"],
    "Rows": [len(single_dataset), len(married_dataset)],
    "States": [single_dataset["State"].nunique(), married_dataset["State"].nunique()],
    "Min Year": [single_dataset["Year"].min(), married_dataset["Year"].min()],
    "Max Year": [single_dataset["Year"].max(), married_dataset["Year"].max()],
    "Duplicate State-Year Rows": [
        single_dataset.duplicated(["State", "Year"]).sum(),
        married_dataset.duplicated(["State", "Year"]).sum(),
    ],
})
validation_summary

## 5. Seaborn visualizations

Every portfolio visualization is generated through Seaborn and exported to `figures/` as a static PNG for direct GitHub rendering.

In [ ]:
plot_real_gdp_growth(annual_change, FIGURES)

In [ ]:
plot_private_employment(US_PRIV, FIGURES)

In [ ]:
plot_cpi_projection(cpi_projection, FIGURES)

In [ ]:
_, latest_married = plot_top_purchasing_power(
    married_dataset,
    constant_dollar_year=2024,
    figure_dir=FIGURES,
)

In [ ]:
plot_top_tax_burden(latest_married, FIGURES)

In [ ]:
plot_purchasing_power_trend(
    married_dataset,
    latest_married,
    constant_dollar_year=2024,
    figure_dir=FIGURES,
)

In [ ]:
plot_income_vs_purchasing_power(
    latest_married,
    constant_dollar_year=2024,
    figure_dir=FIGURES,
)

## 6. Interpretation and limitations

The visualizations should be interpreted as outputs of a standardized scenario, not estimates of observed household disposable income.

- **State income tax is approximate.** The normalized table captures statutory brackets and standard deductions but not every credit, exemption, local tax, or special rule.
- **MHI is not a filing-unit income measure.** Applying a filing status to median household income creates a comparable scenario rather than an observed tax bill.
- **RPP and inflation solve different problems.** RPP handles geographic price-level differences; PCEPI handles changes in the national price level over time.
- **Linear trend gaps are descriptive.** They do not establish causality or the number of jobs/prices that would necessarily have occurred absent the pandemic.